# Fine-tuning Classifier LLM


In [ ]:
# setup - load packages
import pandas as pd
from datasets import Dataset
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

seed = 13

In [46]:
# data in richtige Form
# read data
#data = pd.read_csv("/content/final_data.csv")
preprocessed_data = pd.read_csv("../data/final_data.csv")

# Selecting relevant df columns
preprocessed_data = preprocessed_data[['speech_text', 'party']]
# renaming label category
preprocessed_data = preprocessed_data.rename(columns={'party': 'label'})
# converting to huggingface dataset format
data = Dataset.from_pandas(preprocessed_data)


# splitting into train, test and validation sets
# party data
raw_dataset = data.shuffle(seed=seed)

# 70% train, 15% test, 15% validation data
split = raw_dataset.train_test_split(test_size=0.3, seed=seed)
train_data = split["train"]
text_and_val_data = split["test"]
split = text_and_val_data.train_test_split(test_size=0.5, seed=seed)
test_data = split["train"]
val_data = split["test"]

print(f"Training samples party: {len(train_data)}")
print(f"test samples party: {len(test_data)}")
print(f"Validation samples party: {len(val_data)}")



# data balancing??



Training samples party: 18662
test samples party: 3999
Validation samples party: 3999


In [47]:
# second version - use stopwords on speech to only include meaningful parts of speeches
# preparing stopwords

# importing german stopword list from github
# link: https://github.com/solariz/german_stopwords
with open("german_stopwords_full.txt", "r", encoding="utf-8") as g:
    german_stopwords_full = [line.strip() for line in g if not line.lstrip().startswith(";")]

# maybe list of words to add to stopwords:
more_stopwords = ["damen", "herren", "herr", "kollegen", "kolleginnen", "präsident", "präsidentin"]

# add list of additional words
german_stopwords_full.extend(more_stopwords)

preprocessed_data["tokenized_speeches"] = preprocessed_data["speech_text"].str.lower().str.split()

preprocessed_data['tokenized_speeches'] = preprocessed_data['tokenized_speeches'].apply(lambda tokens: [word for word in tokens if word not in german_stopwords_full and word.isalpha()])

preprocessed_data["speech_text"] = preprocessed_data["tokenized_speeches"].apply(lambda tokens: " ".join(tokens))

preprocessed_data["speech_text"]


0        bankenunion stabilität stabilität erfahrungen ...
1        liebe liebe liebe gäste schön vorsitzenden fin...
2        liebe kollegin thema bankenunion verschlossene...
3        liebe kollege große fraktion geschämt dargebot...
4        liebe vorhin kollegin thema einlagensicherung ...
                               ...                        
26655    reise reise wolfratshausen stadt breiteren öff...
26656    demokratischen antrag union debattieren geschä...
26657    demokratischen unternehmen gekündigt wirtschaf...
26658    liebe frauen deutschland stehen jahr enormen s...
26659    liebe gesetzentwurf hinweisgeberschutz mitte f...
Name: speech_text, Length: 26660, dtype: object

In [48]:
# drop helper column again
preprocessed_data = preprocessed_data.drop(columns="tokenized_speeches", axis = 1)

# converting to huggingface dataset format
data = Dataset.from_pandas(preprocessed_data)


# splitting into train, test and validation sets
# party data
raw_dataset = data.shuffle(seed=seed)

# 70% train, 15% test, 15% validation data
split = raw_dataset.train_test_split(test_size=0.3, seed=seed)
v2_train_data = split["train"]
v2_text_and_val_data = split["test"]
split = v2_text_and_val_data.train_test_split(test_size=0.5, seed=seed)
v2_test_data = split["train"]
v2_val_data = split["test"]

print(f"Training samples party: {len(v2_train_data)}")
print(f"test samples party: {len(v2_test_data)}")
print(f"Validation samples party: {len(v2_val_data)}")

Training samples party: 18662
test samples party: 3999
Validation samples party: 3999


In [49]:
val_data_subset = v2_val_data[:10]

val_data_subset

{'speech_text': ['schwarzer tag vonseiten afd union applaus stellenwert migrationspaket hause bundesregierung verliert abschiebepaket maß mitte massive unverhältnismäßige einschnitte menschenrechte abgelehnten asylbewerbern grundgesetz egal abgelehnte punkte beschließen familien last gelegt ausgereist teils gefährlichen straftätern gefängnissen untergebracht fatal deutsche anwaltverein hingewiesen abschiebehaftsachen wissen korrigierte bundesgerichtshof anforderungen ausreichend beachtet haft deutschland straftat begangen johanna richterin stellte haftentscheidungen amtsgerichte bemerkenswerten umfang rechtswidrig wissen zahl skandal geflüchteten wundert menschenrechtsorganisationen spd gesetz menschenrechtskommissarin europarates schaltete woche gesetz aktivitäten ngos meinungsfreiheit beeinflussen menschenrechtsverteidigerinnen arbeit einschränkung grundrechts drohen haft innenminister christlichen idealen verabschiedet wandelt rechtsstaatlichen spd bundestag gegenwehr verteidigen pa

In [ ]:
# model laden


# quantized?!

In [50]:
# Load Model 
model_name = "bert-base-german-cased" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

print(f"Model parameters: {model.num_parameters():,}")

# Put model in eval mode
model.eval()

# Define possible label names
label_names = ['CDU/CSU', 'SPD', 'GRÜNE', 'FDP', 'AfD', 'LINKE']

#  Tokenize input speeches
inputs = tokenizer(val_data_subset["speech_text"], return_tensors="pt", padding=True, truncation=True)

# Run model to predict basline
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = F.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

# 7. Print prediction examples
for i, pred in enumerate(predictions):
    print(f"Speech {i+1}: predicted party = {label_names[pred]}")
print(logits)

for i, probs in enumerate(probabilities):
    print(f"Speech {i+1} prediction:")
    for label, prob in zip(label_names, probs):
        print(f"  {label:7s}: {prob.item():.2%}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 109,085,958
Speech 1: predicted party = GRÜNE
Speech 2: predicted party = AfD
Speech 3: predicted party = AfD
Speech 4: predicted party = GRÜNE
Speech 5: predicted party = AfD
Speech 6: predicted party = AfD
Speech 7: predicted party = AfD
Speech 8: predicted party = AfD
Speech 9: predicted party = AfD
Speech 10: predicted party = AfD
tensor([[-0.1194, -0.4787,  0.2116,  0.1323,  0.0985, -0.2639],
        [-0.1162, -0.5254,  0.1425,  0.1557,  0.2219, -0.2661],
        [-0.0560, -0.5077,  0.0968,  0.1456,  0.3070, -0.4622],
        [-0.0413, -0.3571,  0.1440,  0.1160,  0.0484, -0.2934],
        [-0.0673, -0.4617,  0.1021,  0.2062,  0.2395, -0.4788],
        [-0.1554, -0.5269,  0.0918,  0.1316,  0.3181, -0.4186],
        [-0.1211, -0.4482,  0.2008,  0.2106,  0.2117, -0.3703],
        [-0.1298, -0.4442,  0.1424,  0.2511,  0.2885, -0.3475],
        [-0.1625, -0.5131,  0.1246,  0.1351,  0.2803, -0.3780],
        [-0.1866, -0.6246,  0.1865,  0.1512,  0.2159, -0.4131]])
Spee

In [ ]:
# define prediction function
# Define possible label names
label_names = ['CDU/CSU', 'SPD', 'GRÜNE', 'FDP', 'AfD', 'LINKE']



def predicting_probs(model, dataset, label_names):
    # Put model in eval mode
    model.eval()

    
    #  Tokenize input speeches
    inputs = tokenizer(val_data_subset["speech_text"], return_tensors="pt", padding=True, truncation=True)

    # Run model to predict basline
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=1)
        predictions = torch.argmax(probabilities, dim=1)

    return probabilities 


def probs_to_preds(model, dataset, label_names):
    # get probabilites
    predicting_probs(model, dataset, label_names)
    # convert to predictions
    predictions = torch.argmax(probabilities, dim=1)

    return predictions



In [13]:
# einmal zero shot ausprobieren mit classifier task
val_data = val_data[:20]


In [ ]:
# fine-tunen



In [ ]:
# validaten



In [ ]:
# compare 2 versions (erinnern!)



In [ ]:
# outcomes test data conf matrix, accuracy



In [ ]:
# save model for futher tests on LLM generated speeches

